# Maker3D — Blender 5.3 Cycles GPU (`OptiX / CUDA`) en Google Colab Pro

Este notebook ejecuta **Blender 5.3.0 headless en GPU real** (`RTX PRO 6000 Blackwell`, `A100`, `L4` o `T4` + **High-RAM**) para hornear y renderizar **`montana_hero360 v05-macro8k`** (`--stage passB`, `--stage render-macros` o `--stage passA`) sin usar CPU lenta.

**Antes de empezar:**
1. En Colab: `Entorno de ejecución -> Cambiar tipo de entorno` -> elige **L4 / A100 / Blackwell PRO 6000** y marca **RAM alta (High-RAM)**.
2. No dupliques archivos: este notebook detecta automáticamente tu carpeta **`Antigravity/Maker3D/Tableros-Catan/pipeline-biomas/montana_hero360/work_v05`** ya sincronizada con Google Drive (`Mi unidad` u `Ordenadores`).

In [ ]:
!nvidia-smi
!free -h
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, pathlib

REPO_DIR = pathlib.Path('/content/Maker3D')
if not REPO_DIR.exists():
    !git clone https://github.com/MaikikiaM/Maker3D.git /content/Maker3D
else:
    !git -C /content/Maker3D pull --ff-only

BLENDER_VER = '5.3.0'
TAR_NAME = f'blender-{BLENDER_VER}-linux-x64.tar.xz'
CACHE_TAR = pathlib.Path('/content/drive/MyDrive/Maker3D/cache') / TAR_NAME
BLENDER_DIR = pathlib.Path(f'/content/blender-{BLENDER_VER}-linux-x64')
BLENDER_BIN = BLENDER_DIR / 'blender'

if not BLENDER_BIN.exists():
    CACHE_TAR.parent.mkdir(parents=True, exist_ok=True)
    if not CACHE_TAR.exists():
        url = f'https://download.blender.org/release/Blender5.3/{TAR_NAME}'
        subprocess.run(['wget', '-q', '--show-progress', '-O', str(CACHE_TAR), url], check=True)
    subprocess.run(['tar', '-xf', str(CACHE_TAR), '-C', '/content'], check=True)

!{BLENDER_BIN} --version
!apt-get -qq update && apt-get -qq install -y libxrender1 libxi6 libxxf86vm1 libxfixes3 libgl1 libxkbcommon0 > /dev/null
!pip install -q opencv-python-headless numpy scipy pillow trimesh

In [ ]:
# Comprobar qué backend GPU activa Blender (debe imprimir GPU:OPTIX o GPU:CUDA)
!{BLENDER_BIN} -b --python-expr "import bpy, sys; sys.path.insert(0, '/content/Maker3D/tools/blender'); import macro_bake_helpers as m; print('ACTIVE:', m.configure_cycles_device(bpy.context.scene, 'gpu'))"

In [ ]:
# Localizar work_v05 YA sincronizado en Drive (sin duplicar carpetas) y copiar a SSD local (/content/work_v05)
import glob, pathlib
REL_WORK = 'Maker3D/Tableros-Catan/pipeline-biomas/montana_hero360/work_v05'
CANDS = [
    f'/content/drive/MyDrive/Antigravity/{REL_WORK}',
    f'/content/drive/MyDrive/{REL_WORK}',
    *sorted(glob.glob(f'/content/drive/Othercomputers/*/Antigravity/{REL_WORK}')),
    *sorted(glob.glob(f'/content/drive/Othercomputers/*/{REL_WORK}')),
]
DRIVE_WORK = next((c for c in CANDS if pathlib.Path(c).exists()), None)
if not DRIVE_WORK:
    raise FileNotFoundError('No se encontró work_v05 en Drive. Revisados: ' + str(CANDS))
print('DRIVE_WORK detectado (sin duplicar):', DRIVE_WORK)
LOCAL_WORK = '/content/work_v05'

!rm -rf {LOCAL_WORK} && mkdir -p {LOCAL_WORK}
!rsync -ah --info=progress2 "{DRIVE_WORK}/" "{LOCAL_WORK}/"

In [ ]:
# (Opcional) Re-ejecutar alineado High-Pass 16-bit si has añadido imágenes nuevas en macro/nb/
!python /content/Maker3D/tools/generative/macro_patch_align.py \
  --macro-dir {LOCAL_WORK}/macro \
  --sigma-hp 6.0

In [ ]:
# Hornear Pass B 8K + renders de auditoría en Cycles OptiX/CUDA
!{BLENDER_BIN} -b --python-exit-code 1 \
  --python /content/Maker3D/tools/blender/catan_hero360_bake.py -- \
  --stl {LOCAL_WORK}/tile_base_talus.stl \
  --cameras {LOCAL_WORK}/cameras.json \
  --out-dir {LOCAL_WORK} \
  --macro-dir {LOCAL_WORK}/macro \
  --bake-res 8192 \
  --normal-res 8192 \
  --rough-res 4096 \
  --bake-samples 32 \
  --render-samples 48 \
  --device gpu \
  --stage passB

In [ ]:
# Sincronizar resultados finales de vuelta a Google Drive
!rsync -ah --info=progress2 \
  {LOCAL_WORK}/tile_master.stl \
  {LOCAL_WORK}/tile_viewer.glb \
  {LOCAL_WORK}/baked_albedo.png \
  {LOCAL_WORK}/baked_normal.png \
  {LOCAL_WORK}/baked_roughness.png \
  {LOCAL_WORK}/audit_6views_pbr.png \
  {LOCAL_WORK}/audit_6views_clay.png \
  {LOCAL_WORK}/audit_ortho.png \
  "{DRIVE_WORK}/"